In [ ]:
import json
import numpy as np

from theia.grids import LatLonHeightGrid
from theia.terrain import elevationAt
from theia.types import AttenuationModel, Point, Polarization, Receiver, Transmitter

In [ ]:
from theia.util import erp_to_power, from_dB


def _parse_receiver(
    rx_spec: dict, integration_time: float, tx_bandwidth: float
) -> Receiver:
    lat = rx_spec["lat"]
    lon = rx_spec["lon"]
    alt = elevationAt(lat, lon)

    bw = rx_spec["bandwidth"] / 1000

    return Receiver(
        id=rx_spec["rx_id"],
        point=Point(lat=lat, lon=lon, alt=alt),
        antenna_height=rx_spec["ahmagl"],
        diameter=1.0,
        cpi_pulses=int(np.floor(integration_time * tx_bandwidth * 1e6)),
        pfa=np.nan,
        min_elevation=np.nan,
        max_elevation=np.nan,
        rotation_time=np.nan,
        bandwidth=bw,
        gain=0.0,
    )


def _parse_transmitter(tx_spec: dict) -> Transmitter:
    lat = tx_spec["lat"]
    lon = tx_spec["lon"]
    alt = elevationAt(lat, lon)

    erp_h = tx_spec["erp_h"] if tx_spec["erp_h"] != "UNDEFINED" else np.nan
    erp_v = tx_spec["erp_v"] if tx_spec["erp_v"] != "UNDEFINED" else np.nan
    assert np.logical_xor(np.isnan(erp_h), np.isnan(erp_v))
    erp = erp_h if not np.isnan(erp_h) else erp_v

    polarization = Polarization.HORIZONTAL
    if tx_spec["pol"] == "H":
        pass
    elif tx_spec["pol"] == "V":
        polarization = Polarization.VERTICAL
    else:
        raise ValueError(f"Unknown polarization {tx_spec['pol']}")

    if tx_spec["horiz_diagr_att"] == "UNDEFINED":
        h_model = None
    else:
        h_values = tx_spec["horiz_diagr_att"]
        if type(h_values) is int:
            h_values = [h_values]
        h_angles = np.linspace(0, 2 * np.pi, len(h_values))
        h_model = AttenuationModel(
            attenuation_table_angles=h_angles,
            attenuation_table_values=h_values,
            polarization=Polarization.HORIZONTAL,
        )

    if tx_spec["vert_diagr_att"] == "UNDEFINED":
        v_model = None
    else:
        v_values = tx_spec["vert_diagr_att"]
        v_angles = np.linspace(-np.pi / 2, np.pi / 2, len(v_values))
        v_model = AttenuationModel(
            attenuation_table_angles=v_angles,
            attenuation_table_values=v_values,
            polarization=Polarization.VERTICAL,
        )

    return Transmitter(
        id=tx_spec["tx_id"],
        point=Point(
            lat=lat,
            lon=lon,
            alt=alt,
        ),
        power=erp_to_power(erp, 0.0, 0.0),
        erp=from_dB(erp),
        antenna_height=tx_spec["ahmagl"],
        antenna_diameter=1.0,
        antenna_gain=0.0,
        frequency=tx_spec["freq"],
        bandwidth=tx_spec["bandwidth"] / 1000.0,
        pulse_width=np.nan,
        polarization=polarization,
        vertical_attenuation=v_model,
        horizontal_attenuation=h_model,
    )

In [ ]:
from theia.data_loading import load_bakom_ukw_transmitters


transmitters = load_bakom_ukw_transmitters()

In [ ]:
import numpy as np


[t for t in transmitters if np.isclose(t.lat, 47.01333)]

In [ ]:
path = "tests/test_data/pcl_coverage/request_single_point.json"

with open(path, "r") as file:
    content = json.load(file)

integration_time = content["integration_time_s"]

transmitters = [_parse_transmitter(tx_spec) for tx_spec in content["transmitters"]]
assert all([t.bandwidth == transmitters[0].bandwidth for t in transmitters])

receivers = [
    _parse_receiver(rx_spec, integration_time, transmitters[0].bandwidth)
    for rx_spec in content["receivers"]
]

grid = LatLonHeightGrid(
    lat_start=content["grid"]["lat_start"],
    lat_stop=content["grid"]["lat_stop"],
    lat_res=content["grid"]["res_y"],
    lon_start=content["grid"]["lon_start"],
    lon_stop=content["grid"]["lon_stop"],
    lon_res=content["grid"]["res_x"],
    height_start=content["grid"]["min_z"],
    height_stop=content["grid"]["max_z"],
    height_res=content["grid"]["res_z"],
)

snr_threshold_dB = content["snr_threshold_dB"]
delay_threshold_us = content["delay_threshold_us"]

In [ ]:
with open("tests/test_data/pcl_coverage/result_single_point.json", "r") as file:
    values_dict = json.load(file)
    min_detectable_rcs = np.array(values_dict)

In [ ]:
point_min_detectable_rcs: list[tuple[Point, float]] = []

for point, value in zip(grid.points, min_detectable_rcs.flatten(), strict=True):
    point_min_detectable_rcs.append(
        (Point(lat=point[0], lon=point[1], alt=point[2]), float(value))
    )

In [ ]:
from theia.detection.pcl import PclDetector


detector = PclDetector(snr_threshold=snr_threshold_dB)

In [ ]:
from theia.detection.pcl import calculate_minimum_detectable_rcs
from theia.types import ConstantRcsModel, Target, Velocity


assert len(receivers) == 1
rx = receivers[0]
tx = transmitters[0]

p, rcs_true = point_min_detectable_rcs[0]
snr_true = 15.683
rcs_true = 0.855
tgt = Target(
    id=0,
    point=p,
    cross_section_model=ConstantRcsModel(rcs=1.0),
    velocity=Velocity(vx=0.0, vy=0.0, vz=0.0),
)


snr_over_rcs, bistatic_range, doppler = detector.calculate_raw_measurement(
    rx,
    tx,
    tgt,
)
rcs_calc = calculate_minimum_detectable_rcs(snr_over_rcs, snr_threshold_dB)

In [ ]:
rcs_true, rcs_calc

In [ ]:
snr_diff = np.log10(rcs_true) * 10.0 - np.log10(rcs_calc) * 10.0
float(snr_diff), float(snr_over_rcs - snr_true)

In [ ]:
from theia.openburst_client import OpenburstClient


client = OpenburstClient("localhost", debugging=True)

result = client.calculate_min_det_rcs_coverage(
    rx,
    tx,
    grid,
    snr_threshold=snr_threshold_dB,
    delay_threshold=delay_threshold_us,
    integration_time=integration_time,
)
result

In [ ]:
result[0].flatten()[0] - rcs_true